# AI-ML Assignment – 2
## Topic: Customer Churn Prediction using Logistic Regression

### Student Details
- **Name**: Arsh Baktoo  
- **Registration No**: 23BCE10430  
- **Application Number**: IN26010763  
- **Batch Number**: 2(B)  
- **Email ID**: arshbaktoo@gmail.com  

---

## Task 1: Data Understanding (2 Marks)

1. **Load the dataset using Pandas.**
2. **Display the first five records.**
3. **Identify:**
   - Numerical features
   - Categorical features
   - Target variable (`Churn`)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Set aesthetic plot style
sns.set_theme(style="whitegrid")

In [2]:
# 1. Load the dataset using Pandas
df = pd.read_csv('churn.csv')
print(f"Dataset loaded successfully. Total Records: {df.shape[0]}, Total Columns: {df.shape[1]}")

Dataset loaded successfully. Total Records: 7043, Total Columns: 21


In [3]:
# 2. Display the first five records
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [4]:
# Clean TotalCharges type for identification
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')

numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
categorical_features = [col for col in df.columns if df[col].dtype == 'object' and col not in ['customerID', 'Churn']]
target_variable = 'Churn'

print("=== Identification of Variables ===")
print(f"Numerical Features   : {numerical_features}")
print(f"Categorical Features : {categorical_features}")
print(f"Target Variable      : {target_variable}")

=== Identification of Variables ===
Numerical Features   : ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
Categorical Features : ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Target Variable      : Churn


--- 
## Task 2: Data Preprocessing (2 Marks)

Perform the following:
- Check for missing values and impute if necessary.
- Encode categorical variables using One-Hot Encoding.
- Apply Feature Scaling (`StandardScaler`).
- Split the dataset into 80% training and 20% testing sets.

In [5]:
# 1. Check missing values
missing_vals = df.isnull().sum()
print(f"Missing values in TotalCharges: {missing_vals['TotalCharges']}")

# Impute missing values with column median
median_val = df['TotalCharges'].median()
df['TotalCharges'].fillna(median_val, inplace=True)
print(f"Imputed missing values with median: {median_val:.2f}")

Missing values in TotalCharges: 11
Imputed missing values with median: 1397.47


In [6]:
# Drop customerID identifier
if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)

# Encode Target variable (Yes -> 1, No -> 0)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
print("Target 'Churn' distribution:")
print(df['Churn'].value_counts())

# Separate features and target
X_raw = df.drop('Churn', axis=1)
y = df['Churn']

# 2. One-Hot Encoding for categorical features
X_encoded = pd.get_dummies(X_raw, drop_first=True)
print(f"\nTotal Encoded Features: {X_encoded.shape[1]}")

Target 'Churn' distribution:
0    5174
1    1869
Name: Churn, dtype: int64

Total Encoded Features: 30


In [7]:
# 3. Split dataset into 80% Train and 20% Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Features (X) shape: {X_encoded.shape}")
print(f"Target (y) shape  : {y.shape}")
print(f"X_train shape     : {X_train.shape} [80%]")
print(f"X_test shape      : {X_test.shape} [20%]")

Features (X) shape: (7043, 30)
Target (y) shape  : (7043,)
X_train shape     : (5634, 30) [80%]
X_test shape      : (1409, 30) [20%]


--- 
## Task 3: Model Development (3 Marks)

- Build a **Logistic Regression** classifier to predict customer churn.
- Train the model on scaled training data.
- Predict churn labels and probabilities for the test dataset.

In [8]:
# Build and train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)

print("Logistic Regression Model trained successfully.")
print(f"Model Intercept: {model.intercept_[0]:.4f}")

Logistic Regression Model trained successfully.
Model Intercept: -0.6601


In [9]:
# Predict on test dataset
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]

# Create sample prediction table
predictions_df = pd.DataFrame({
    'Actual Churn': y_test.values,
    'Predicted Churn': y_pred,
    'Churn Probability': np.round(y_pred_proba, 4)
})
predictions_df.head()

,Actual Churn,Predicted Churn,Churn Probability
0,0,0,0.0384
1,0,0,0.1384
2,0,0,0.0812
3,0,0,0.3125
4,0,0,0.0241


--- 
## Task 4: Model Evaluation (2 Marks)

Evaluate the model using:
- Accuracy, Precision, Recall, F1-Score, ROC-AUC Score
- Confusion Matrix Heatmap & ROC Curve
- Performance Observations.

In [10]:
# Compute Evaluation Metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("=== Performance Metrics ===")
print(f"Accuracy  : {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision : {prec:.4f} ({prec*100:.2f}%)")
print(f"Recall    : {rec:.4f} ({rec*100:.2f}%)")
print(f"F1-Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Churn (0)', 'Churn (1)']))

=== Performance Metrics ===
Accuracy  : 0.8070 (80.70%)
Precision : 0.6584 (65.84%)
Recall    : 0.5668 (56.68%)
F1-Score  : 0.6092
ROC-AUC   : 0.8416

Classification Report:
              precision    recall  f1-score   support

No Churn (0)       0.85      0.89      0.87      1035
   Churn (1)       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [11]:
# Visualizations: Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=300)

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['No Churn', 'Churn'], yticklabels=['No Churn', 'Churn'])
axes[0].set_title('Confusion Matrix', fontsize=13, fontweight='bold', pad=12)
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('Actual Label', fontsize=11)

# 2. ROC Curve Plot
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, color='#1f77b4', linewidth=2.5, label=f'Logistic Regression (AUC = {roc_auc:.4f})')
axes[1].plot([0, 1], [0, 1], color='red', linestyle='--', linewidth=2, label='Random Chance (AUC = 0.50)')
axes[1].set_title('Receiver Operating Characteristic (ROC) Curve', fontsize=13, fontweight='bold', pad=12)
axes[1].set_xlabel('False Positive Rate (1 - Specificity)', fontsize=11)
axes[1].set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=11)
axes[1].legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.show()

### Performance Observations

1. **Strong Overall Accuracy & Discrimination ($80.70\%$, $\text{ROC-AUC} = 0.8416$):** The Logistic Regression model demonstrates solid predictive performance, separating churners from non-churners effectively.
2. **Imbalance Impact on Recall ($56.68\%$ vs Precision $65.84\%$):** Because non-churning customers form the majority class (~73.5%), default probability thresholding (0.5) prioritizes overall accuracy, leading to moderate recall on true churners.
3. **Key Churn Predictors:** Short customer tenure, month-to-month contracts, higher monthly charges, and fiber optic internet subscriptions serve as the strongest positive predictors of customer churn.

--- 
## Task 5: Conclusion (1 Mark)

Write a 100–150 word conclusion covering:
- Key findings
- Factors affecting customer churn
- One limitation of Logistic Regression for this problem

### Conclusion

This project implemented a Logistic Regression model to predict customer churn in a telecommunications service provider. The model demonstrated robust predictive capability, achieving an accuracy of 80.70% and an ROC-AUC score of 0.8416 on the test dataset. Feature analysis reveals that customer tenure, contract length (month-to-month contracts), monthly charges, and internet service type are the primary drivers influencing churn probability. Customers with short tenure and month-to-month subscriptions exhibit significantly higher churn risk. A key limitation of standard Logistic Regression in customer churn prediction is its linear decision boundary assumption and sensitivity to class imbalance. Since retaining churn-prone customers is vital, future improvements should incorporate threshold tuning, class weighting, or non-linear ensemble models (e.g., Random Forest or XGBoost) to boost recall for high-risk customers.